In [ ]:
!pip install -r requirements.txt -q
!pip install kaggle -q

In [ ]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# CUDA library paths (RunPod / Colab / lokaal)
try:
    import nvidia
    lib_dirs = [os.path.join(nvidia.__path__[0], d, 'lib')
                for d in os.listdir(nvidia.__path__[0])
                if os.path.isdir(os.path.join(nvidia.__path__[0], d, 'lib'))]
    os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
except ImportError:
    pass

for p in ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64', '/usr/lib/x86_64-linux-gnu']:
    if os.path.isdir(p):
        os.environ['LD_LIBRARY_PATH'] = p + ':' + os.environ.get('LD_LIBRARY_PATH', '')

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU actief: {[g.name for g in gpus]}')
else:
    print('Geen GPU gevonden, training draait op CPU.')

print(f'TensorFlow {tf.__version__} | Apparaten: {[d.device_type for d in tf.config.list_physical_devices()]}')

In [ ]:
import json as _json, zipfile

# Laad env bestand (probeert runpod.env eerst, dan .env)
for env_path in ['runpod.env', '.env']:
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, val = line.split('=', 1)
                    os.environ[key.strip()] = val.strip()
        print(f'{env_path} geladen')
        break

# Kaggle credentials instellen vanuit environment
kaggle_dir = os.path.expanduser('~/.kaggle')
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')

if not os.path.exists(kaggle_json_path):
    kaggle_user = os.environ.get('KAGGLE_USERNAME', '')
    kaggle_key = os.environ.get('KAGGLE_KEY', '')
    if kaggle_user and kaggle_key:
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(kaggle_json_path, 'w') as f:
            _json.dump({'username': kaggle_user, 'key': kaggle_key}, f)
        os.chmod(kaggle_json_path, 0o600)
        print('kaggle.json aangemaakt')

# Download en extract data
data_dir = 'data'
zip_path = os.path.join(data_dir, 'neural-estate-2026.zip')

if not os.path.exists(os.path.join(data_dir, 'train.csv')):
    os.makedirs(data_dir, exist_ok=True)
    !kaggle competitions download -c neural-estate-2026 -p {data_dir}
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)
    print(f'Data gedownload en uitgepakt naar {data_dir}/')
else:
    print('Data bestaat al, download overgeslagen')

print(f'Bestanden in data/: {os.listdir(data_dir)}')

# Deeplearning Opdracht 1: Neural Estate!

### Hoofdstuk 1: inleiding.

Deze notebook onderzoekt hoe we huizenprijzen kunnen voorspellen met deep learning op 2 databronnen: tabulaire kenmerken en afbeeldingen.
Het doel is om systematisch te vergelijken welke aanpak het beste werkt binnen een kleine dataset, en om op basis daarvan een onderbouwde eindkeuze te maken.

We bouwen en evalueren 4 modelstrategieen:

Fully-connected netwerk op tabulaire data, custom CNN op beelddata, transfer learning met EfficientNet80 en multimodaal model dat tabulaire en visuele features combineert.

We hebben het notebook zo opgebouwd als een volledige ML-pipeline: data inladen, EDA, feature engineering, modeltraining, validatie, interpretatie van resultaten en de export van submissions naar kaggle.

Naast modelprestaties leggen wij de nadruk ook op reproduceerbaarheid en transparantie van keuzes, zodat de uitkomsten inhoudelijk en technisch verdedigbaar zijn.

### Hoofdstuk 2: imports, setup en functions.

**Zo run je dit notebook:**
1. Gebruik een GPU-omgeving (RunPod, Colab)
2. Maak een `runpod.env` bestand aan in de project root met je Kaggle credentials:
   ```
   KAGGLE_USERNAME=jouw_username
   KAGGLE_KEY=jouw_api_key
   ```
3. Run alle cellen in volgorde - dependencies, GPU en data worden automatisch geregeld

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import cv2
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns

from sklearn.metrics import mean_absolute_percentage_error
from scipy.optimize import minimize
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import RobustScaler

from tensorflow.keras import callbacks, layers, losses, metrics, models, optimizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

optuna.logging.set_verbosity(optuna.logging.WARNING)
%matplotlib inline

random_seed = 42
SEED = random_seed
np.random.seed(random_seed)
tf.random.set_seed(random_seed)
IMG_SIZE = 380

print(f"TF {tf.__version__} | NumPy {np.__version__} | GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
#functies: 

def load_housing_data():
    """
    Laadt de train en test dataframes in vanuit de vaste mappenstructuur.
    Voegt automatisch een kolom 'image_path' toe met het volledige pad naar de foto.
    """
    # Verkrijg het pad van de map waar dit notebook staat (repo root)
    base_path = os.path.join(os.getcwd(), 'data')
    
    train_csv_path = os.path.join(base_path, 'train.csv')
    test_csv_path = os.path.join(base_path, 'test.csv')
    train_img_dir = os.path.join(base_path, 'Train')
    test_img_dir = os.path.join(base_path, 'Test')

    # Inladen van de CSV bestanden
    if not os.path.exists(train_csv_path):
        raise FileNotFoundError(f"Kon train.csv niet vinden op: {train_csv_path}. Check je mappenstructuur!")

    df_train = pd.read_csv(train_csv_path)
    df_test = pd.read_csv(test_csv_path)

    
    def build_path(house_id, folder):
        return os.path.join(folder, f"{int(house_id)}.jpg")

    # Kolommen toevoegen met de paden naar de afbeeldingen
    df_train['image_path'] = df_train['House ID'].apply(lambda x: build_path(x, train_img_dir))
    df_test['image_path'] = df_test['House ID'].apply(lambda x: build_path(x, test_img_dir))

    print(f" Data succesvol geladen vanuit {base_path}")
    print(f"Aantal train samples: {len(df_train)}")
    print(f"Aantal test samples: {len(df_test)}")
    
    return df_train, df_test

def plot_sample_images(df, num_samples=3):
    """
    Selecteert willekeurige huizen uit het dataframe en toont de bijbehorende foto's
    met de belangrijkste metadata als titel.
    """
    samples = df.sample(num_samples, random_state = random_seed)
    
    plt.figure(figsize=(20, 10))
    
    for i, (idx, row) in enumerate(samples.iterrows()):
        img_path = row['image_path']
        
        # Inladen met OpenCV
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
            
            plt.subplot(1, num_samples, i + 1)
            plt.imshow(img)
            
            # Titel met prijs en kenmerken
            title = (f"House ID: {int(row['House ID'])}\n"
                     f"Price: ${row['Price']:,}\n"
                     f"Beds: {row['Bedrooms']}, Baths: {row['Bathrooms']}\n"
                     f"Area: {row['Area']} sqft")
            
            plt.title(title, fontsize=10)
            plt.axis('off')
        else:
            print(f"Waarschuwing: Afbeelding niet gevonden op {img_path}")
            
    plt.tight_layout()
    plt.show()

def plot_metadata_distributions(df):
    """
    Maakt histogrammen en boxplots voor de numerieke variabelen 
    om de spreiding en uitschieters te visualiseren.
    """
    features = ['Bedrooms', 'Bathrooms', 'Area', 'Price']
    
    fig, axes = plt.subplots(len(features), 2, figsize=(15, 12))
    
    for i, col in enumerate(features):
        # Histogram voor de verdeling
        sns.histplot(df[col], kde=True, ax=axes[i, 0], color='skyblue')
        axes[i, 0].set_title(f'Verdeling van {col}')
        
        # Boxplot voor uitschieters (outliers)
        sns.boxplot(x=df[col], ax=axes[i, 1], color='lightcoral')
        axes[i, 1].set_title(f'Outliers in {col}')
        
    plt.tight_layout()
    plt.show()

def plot_geographic_data(df):
    """
    Visualiseert de locatie van de huizen op basis van Lat/Long,
    waarbij de kleur de prijs aangeeft.
    """
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(df['Longitude'], df['Latitude'], 
                          c=df['Price'], cmap='viridis', 
                          alpha=0.6, s=df['Area']/100) 
    
    plt.colorbar(scatter, label='Prijs ($)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Geografische spreiding van huizen (Kleur = Prijs, Grootte = Oppervlakte)')
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_correlation_matrix(df):
    """
    Toont een heatmap van de correlaties tussen de numerieke variabelen.
    Dit kan helpen om te begrijpen welke kenmerken sterk samenhangen met de prijs.
    """
    numeric_df = df.select_dtypes(include=[np.number])
    corr_matrix = numeric_df.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title('Correlatiematrix van Numerieke Variabelen')
    plt.show()

def cheap_vs_expensive(df):
    """
    Vergelijkt de goedkoopste en duurste huizen op basis van hun kenmerken en afbeeldingen.
    """
    cheapest = df.nsmallest(1, 'Price').iloc[0]
    most_expensive = df.nlargest(1, 'Price').iloc[0]
    
    # Toon afbeeldingen
    plot_sample_images(pd.DataFrame([cheapest, most_expensive]), num_samples=2)

import time

class TrainProgress:
    """Toont voortgang tijdens K-Fold training met tijdschatting."""
    def __init__(self, model_name, n_folds, n_seeds=1):
        self.model_name = model_name
        self.n_folds = n_folds
        self.n_seeds = n_seeds
        self.total = n_folds * n_seeds
        self.done = 0
        self.fold_times = []
        self.start = time.time()
        self.fold_start = None
        print(f"\n{'='*60}")
        print(f"  {model_name} | {n_folds} folds" + (f" x {n_seeds} seeds" if n_seeds > 1 else ""))
        print(f"{'='*60}")

    def begin_fold(self, fold, seed=None):
        self.fold_start = time.time()
        label = f"Fold {fold}/{self.n_folds}"
        if seed is not None:
            label += f" seed {seed}"
        pct = self.done / self.total * 100
        n_done = int(pct // 5)
        bar = '#' * n_done + '-' * (20 - n_done)
        eta = ""
        if self.fold_times:
            avg = sum(self.fold_times) / len(self.fold_times)
            remaining = avg * (self.total - self.done)
            eta = f" | ETA: {remaining/60:.0f}min"
        print(f"\n[{bar}] {pct:5.1f}%{eta}")
        print(f"  > {label}", end="", flush=True)

    def end_fold(self, fold_mape=None):
        elapsed = time.time() - self.fold_start
        self.fold_times.append(elapsed)
        self.done += 1
        msg = f" -- {elapsed:.0f}s"
        if fold_mape is not None:
            msg += f" -- MAPE: {fold_mape:.4f}"
        print(msg)

    def summary(self, final_mape):
        total_time = time.time() - self.start
        bar = '#' * 20
        print(f"\n[{bar}] 100.0%")
        print(f"  DONE: {self.model_name} in {total_time/60:.1f} min | OOF MAPE: {final_mape:.4f}")
        print(f"{'='*60}")


### Hoofdstuk 2:  Exploratieve Data Analyse

In [ ]:
#laad data:
df_train, df_test = load_housing_data()

display(df_train.head())

In [ ]:
#statistieken:
train_numeric = df_train.select_dtypes(include=[np.number])
display(train_numeric.describe())
display(train_numeric.info())

In [ ]:
# plot random voorbeeldafbeeldingen met metadata
plot_sample_images(df_train, num_samples=3)

In [ ]:
# plot metadata distributies
plot_metadata_distributions(df_train)

In [ ]:
# meer plots:
plot_geographic_data(df_train)

#correlatiematrix:
plot_correlation_matrix(df_train)

In [ ]:
# duurste vs goedkoopste huis:
cheap_vs_expensive(df_train)

#### Belangrijkste vondsten en interpretaties EDA (Exploratory Data Analysis)
Uit de grondige en systematische analyse van de dataset zijn diverse dingen naar voren gekomen die de koers voor onze verdere netwerken bepalen:
- **Correlaties & Patronen**: Oppervlakte (area), aantal badkamers en slaapkamers hebben logischerwijs de sterkste positieve correlatie met de target variabele 'price'. De heatmap laat tevens multicollineariteit zien (zoals verwacht hangt area en aantal slaapkamers samen). Daarom helpt het netwerk later om wellicht interactie-features te bouwen. We zien ook duidelijke geografische clusters waar huizen simpelweg meer waard zijn door hun ligging.
- **Datakwaliteit en Beelden**: Bijna elke foto in de set hanteert dezelfde 4-grid collage (badkamer, slaapkamer, keuken, voorkant). Dit is enorm wenselijk: een CNN netwerk hoeft minder te worstelen om focus-gebieden te vinden gezien de constante beeldopbouw. Wel voegt deze framing uiteraard veel hoge dichtheid 'ruis' toe aan randen tussen de kaders. 
- **Bias & Outliers**: De spreiding van huizen is alles behalve normaal verdeeld: enorm right-skewed in de distributie waarbij enkele gigantische villa's miljoenen dollars waard zijn. Om te voorkomen dat RMSE / L2 gebaseerde theorieen keiharde bias op deze paar grote foute voorspellingen krijgen, zullen we bij de architectuur de target-waarde logaritmisch moeten transformeren naar `log1p(Price)` in combinatie met een loss funcitie als Huber die L1/L2 combineert als schild tegen outliers.

### Hoofdstuk 3: Fully-connected neuraal netwerk (tabulaire data)

In dit hoofdstuk gebruiken we alleen de tabulaire kenmerken om de huizenprijs te voorspellen.
De afbeelding wordt hier bewust niet gebruikt.

#### 3.1 Data voorbereiding en feature engineering

In [ ]:
target_col = 'Price'
id_col = 'House ID'
excluded_cols = {target_col, id_col, 'image_path'}

df_train['Area_per_Bedroom'] = df_train['Area'] / df_train['Bedrooms'].clip(lower=1)
df_train['Area_per_Bathroom'] = df_train['Area'] / df_train['Bathrooms'].clip(lower=1)
df_train['Bed_Bath_Ratio'] = df_train['Bedrooms'] / df_train['Bathrooms'].clip(lower=1)
df_train['Total_Rooms'] = df_train['Bedrooms'] + df_train['Bathrooms']
df_train['Area_x_Rooms'] = df_train['Area'] * df_train['Total_Rooms']
df_train['Log_Area'] = np.log1p(df_train['Area'])

df_test['Area_per_Bedroom'] = df_test['Area'] / df_test['Bedrooms'].clip(lower=1)
df_test['Area_per_Bathroom'] = df_test['Area'] / df_test['Bathrooms'].clip(lower=1)
df_test['Bed_Bath_Ratio'] = df_test['Bedrooms'] / df_test['Bathrooms'].clip(lower=1)
df_test['Total_Rooms'] = df_test['Bedrooms'] + df_test['Bathrooms']
df_test['Area_x_Rooms'] = df_test['Area'] * df_test['Total_Rooms']
df_test['Log_Area'] = np.log1p(df_test['Area'])

# Extra interactie features
for _df in [df_train, df_test]:
    _df['Rooms_per_Area'] = _df['Total_Rooms'] / _df['Area'].clip(lower=1)
    _df['Bath_per_Bed'] = _df['Bathrooms'] / _df['Bedrooms'].clip(lower=1)
    _df['Area_log_sq'] = np.log1p(_df['Area']) ** 2
    _df['Bed_x_Bath'] = _df['Bedrooms'] * _df['Bathrooms']
    _df['Area_per_Room'] = _df['Area'] / _df['Total_Rooms'].clip(lower=1)

# Locatie features
for _df in [df_train, df_test]:
    _df['Lat_sin'] = np.sin(np.radians(_df['Latitude']))
    _df['Lat_cos'] = np.cos(np.radians(_df['Latitude']))
    _df['Lon_sin'] = np.sin(np.radians(_df['Longitude']))
    _df['Lon_cos'] = np.cos(np.radians(_df['Longitude']))
    _df['Lat_Lon'] = _df['Latitude'] * _df['Longitude']
    _df['Area_sq'] = _df['Area'] ** 2
    _df['Luxury'] = _df['Bathrooms'] * _df['Area'] / _df['Bedrooms'].clip(lower=1)

# KMeans locatie clusters
N_CLUSTERS = 6
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=random_seed, n_init=10)
kmeans.fit(df_train[['Latitude', 'Longitude']].values)
for _df in [df_train, df_test]:
    coords = _df[['Latitude', 'Longitude']].values
    _df['Location_Cluster'] = kmeans.predict(coords)
    for ci, center in enumerate(kmeans.cluster_centers_):
        _df[f'Dist_Cluster_{ci}'] = np.sqrt((coords[:,0]-center[0])**2 + (coords[:,1]-center[1])**2)

feature_cols = [
    col for col in df_train.columns
    if col not in excluded_cols and pd.api.types.is_numeric_dtype(df_train[col])
]

X = df_train[feature_cols].copy()
y_raw = df_train[target_col].astype('float32').copy()
y = np.log1p(y_raw).astype('float32')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=random_seed
)
_, _, y_train_raw, y_val_raw = train_test_split(
    X, y_raw, test_size=0.2, random_state=random_seed
)

median_values = X_train.median(numeric_only=True)
X_train = X_train.fillna(median_values)
X_val = X_val.fillna(median_values)
X_test_tab = df_test[feature_cols].copy().fillna(median_values)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_tab)

print('Gebruikte features:', feature_cols)
print('Train shape:', X_train_scaled.shape, '| Val shape:', X_val_scaled.shape)
print(f'Target is nu log-getransformeerd: log1p(Price), range [{y.min():.2f}, {y.max():.2f}]')

#### 3.2 Theoretische onderbouwing van Architectuurkeuzes FC layer
- **Target Variabele:** We voorspellen via `log1p(Price)`. Door een log-transformatie te doen verandert een asymmetrische en exponentiele theorie-distributie van rijkdom (enkele miljoenen-huizen als uitschieter) naar iets wat dichter bij een normale verdeling komt. Met name de *Huber-loss* in combinatie met deze transformatie pinaliseert relatieve afwijkingen (wat logischer is: een fout van $10k op een klein huis is veel erger dan op een kasteel). Dit moet de MAPE direct verlagen.
- **Architectuur & Diepte:** Dense lagen aflopend in architectuur (256, 128, 64). Lagen van de vorm 2^n blijken in de literatuur goed te werken wegens hoe matrix multiplicaties geoptimaliseerd zijn, en de taps toelopende ('funnel') structuur dwingt het netwerk the extraheren naar steeds de belangrijkste abstracte kenmerken naarmate we richting het voorspellings-getal komen. 
- **Regularisatie:** Ik pas een `Dropout` (0.3 of dynamisch bepaald door optuna) en `BatchNormalization` toe na elke laag. De kleine dataset grootte zal ons netwerk extreem snel laten overfitten, BatchNorm strijkt hierbij de internal covariate shift glad waardoor gradients stabiel blijven in trainen.
- **Lossfunctie:** Naast de target-log pas ik specifiek de *Huber-loss* functie toe in plaats van stanaard MSE. Huber wisselt soepel tussen L1 en L2 loss en is daardoor net even iets robuuster tegen uitschieters die binnen in de dataset mogelijk nog zitten na opschoning.
- **Hyperparameter-optimalisatie:** Hyperparameters deels via de hand maar ook iteratief afgesteld met `Optuna`, dat een robuuster alternatief biedt voor willekeurige grid-searches bij neuronen.

In [ ]:
# Optuna hyperparameter zoektocht met KFold validatie
N_FOLDS = 7
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=random_seed)

def create_fc_model(trial, input_dim):
    n_layers = trial.suggest_int('n_layers', 2, 6)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)

    model = models.Sequential(name='fc_tabular_regressor')
    model.add(layers.Input(shape=(input_dim,), name='tabular_input'))

    for i in range(n_layers):
        u = trial.suggest_int(f'units_l{i}', 64, 768, step=32)
        model.add(layers.Dense(u, activation='relu'))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation='linear', name='price_output'))
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=losses.Huber(),
        metrics=[metrics.MeanAbsoluteError(name='mae')]
    )
    return model

# Optuna op eerste fold
first_tr, first_va = next(iter(kf.split(X)))
sc_opt = RobustScaler()
X_tr_opt = sc_opt.fit_transform(X.values[first_tr])
X_va_opt = sc_opt.transform(X.values[first_va])

def optuna_fc_objective(trial):
    tf.random.set_seed(random_seed)
    model = create_fc_model(trial, X_tr_opt.shape[1])
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    es = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    rlr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
    model.fit(X_tr_opt, y.values[first_tr],
              validation_data=(X_va_opt, y.values[first_va]),
              epochs=200, batch_size=batch_size, callbacks=[es, rlr], verbose=0)
    pred_log = model.predict(X_va_opt, verbose=0).flatten()
    pred_price = np.expm1(pred_log)
    return mean_absolute_percentage_error(y_raw.values[first_va], pred_price)

print("\nOptuna zoekt FC hyperparameters (50 trials)...")
study_fc = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=random_seed))
study_fc.optimize(optuna_fc_objective, n_trials=50, show_progress_bar=True)
print(f"\nBeste MAPE: {study_fc.best_value:.4f}")
print(f"Parameters: {study_fc.best_params}")

In [ ]:
# K-Fold training met beste parameters + seed averaging
best = study_fc.best_params
N_SEEDS = 3
oof_fc_log = np.zeros(len(X))
test_fc_log = np.zeros(len(df_test))

progress = TrainProgress('FC Netwerk', N_FOLDS, N_SEEDS)
for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    fold_oof = np.zeros(len(va_idx))
    fold_test = np.zeros(len(df_test))

    for seed_i in range(N_SEEDS):
        progress.begin_fold(fold+1, seed=seed_i+1)
        tf.random.set_seed(random_seed + fold * 100 + seed_i)

        sc_f = RobustScaler()
        X_tr = sc_f.fit_transform(X.values[tr_idx])
        X_va = sc_f.transform(X.values[va_idx])
        X_te = sc_f.transform(df_test[feature_cols].fillna(X.median()))

        fc_model = models.Sequential(name='fc_tabular_regressor')
        fc_model.add(layers.Input(shape=(X_tr.shape[1],), name='tabular_input'))
        for i in range(best['n_layers']):
            fc_model.add(layers.Dense(best[f'units_l{i}'], activation='relu'))
            fc_model.add(layers.BatchNormalization())
            fc_model.add(layers.Dropout(best['dropout']))
        fc_model.add(layers.Dense(1, activation='linear', name='price_output'))
        fc_model.compile(
            optimizer=optimizers.Adam(learning_rate=best['lr']),
            loss=losses.Huber(),
            metrics=[metrics.MeanAbsoluteError(name='mae')]
        )

        es = callbacks.EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True)
        rlr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
        fc_model.fit(X_tr, y.values[tr_idx], validation_data=(X_va, y.values[va_idx]),
                     epochs=800, batch_size=best.get('batch_size', 32),
                     callbacks=[es, rlr], verbose=0)

        fold_oof += fc_model.predict(X_va, verbose=0).flatten() / N_SEEDS
        fold_test += fc_model.predict(X_te, verbose=0).flatten() / N_SEEDS
        fold_mape = mean_absolute_percentage_error(y_raw.values[va_idx], np.expm1(fold_oof * N_SEEDS / (seed_i+1)))
        progress.end_fold(fold_mape)

    oof_fc_log[va_idx] = fold_oof
    test_fc_log += fold_test / N_FOLDS

oof_fc_price = np.expm1(oof_fc_log)
test_fc_price = np.expm1(test_fc_log)
fc_mape = mean_absolute_percentage_error(y_raw, oof_fc_price)
progress.summary(fc_mape)

#### 3.2 K-Fold training

In [ ]:
print(f"FC OOF MAPE: {fc_mape:.4f}")

In [ ]:
# Scatter plot: werkelijke vs voorspelde prijzen (OOF)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_raw, oof_fc_price, alpha=0.5, s=30)
ax.plot([y_raw.min(), y_raw.max()], [y_raw.min(), y_raw.max()], 'r--', lw=2)
ax.set_xlabel('Werkelijke Prijs ($)')
ax.set_ylabel('Voorspelde Prijs ($)')
ax.set_title(f'FC Model: Werkelijk vs Voorspeld (OOF MAPE: {fc_mape:.4f})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(f"FC OOF MAPE: {fc_mape:.4f}")

#### 3.3 Submission

In [ ]:
submission_fc = pd.DataFrame({
    'House ID': df_test[id_col].astype(int),
    'Price': np.maximum(test_fc_price, 0)
})
os.makedirs('submissions', exist_ok=True)
submission_fc.to_csv('submissions/submission_fc_nn.csv', index=False)
print('FC submission opgeslagen: submissions/submission_fc_nn.csv')
display(submission_fc.head())

### Hoofdstuk 4: Convolutioneel Neuraal Netwerk (CNN)

In dit hoofdstuk trainen we een CNN op de huisafbeeldingen om visuele kenmerken
(afwerkingsniveau, grootte, staat van onderhoud) te leren herkennen als prijsindicator.

In [ ]:
def load_images(df, img_size=IMG_SIZE):
    images = []
    for path in df['image_path']:
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
        images.append(img)
    return preprocess_input(np.array(images, dtype='float32'))

print("Laden van afbeeldingen...")
X_images_all = load_images(df_train)
X_images_test = load_images(df_test)
print(f"Train: {X_images_all.shape} | Test: {X_images_test.shape}")

train_datagen = ImageDataGenerator(
    rotation_range=25, width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True, zoom_range=0.2, brightness_range=[0.7, 1.3],
    shear_range=0.1, channel_shift_range=20.0, fill_mode='reflect'
)

def denormalize_efficientnet(img):
    return np.clip((img + 1.0) / 2.0, 0, 1)

# Augmentatie voorbeelden
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_img = X_images_all[0:1]
axes[0, 0].imshow(denormalize_efficientnet(sample_img[0]))
axes[0, 0].set_title('Origineel'); axes[0, 0].axis('off')
for i in range(1, 8):
    aug_img = train_datagen.random_transform(sample_img[0])
    r, c = divmod(i, 4)
    axes[r, c].imshow(denormalize_efficientnet(aug_img))
    axes[r, c].set_title(f'Aug {i}'); axes[r, c].axis('off')
plt.suptitle('Data-augmentatie voorbeelden', fontsize=14)
plt.tight_layout()
plt.show()

#### 4.2 CNN Architectuur
Custom CNN met oplopende filters (32-64-128-256), GlobalAveragePooling, BatchNorm en Dropout.
Data augmentatie compenseert de kleine dataset. L2 regularisatie voorkomt overfitten.

In [ ]:
def build_cnn_model(img_size=IMG_SIZE):
    model = models.Sequential(name='cnn_image_regressor')
    
    model.add(layers.Input(shape=(img_size, img_size, 3)))
    
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                            kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                            kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                            kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same',
                            kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation='relu',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation='relu',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(1, activation='linear'))
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=3e-4),
        loss=losses.Huber(),
        metrics=[metrics.MeanAbsoluteError(name='mae')]
    )
    return model

cnn_model = build_cnn_model()
cnn_model.summary()

#### 4.3 CNN K-Fold training

In [ ]:
def augmented_generator(X_imgs, y_targets, batch_size=16, datagen=train_datagen):
    n = len(X_imgs)
    while True:
        indices = np.random.permutation(n)
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_idx = indices[start:end]
            batch_x = np.array([datagen.random_transform(X_imgs[i]) for i in batch_idx])
            batch_y = y_targets[batch_idx]
            yield batch_x, batch_y

# CNN K-Fold training
oof_cnn_log = np.zeros(len(df_train))
test_cnn_log = np.zeros(len(df_test))
cnn_batch_size = 32

progress = TrainProgress('CNN', N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(df_train)))):
    progress.begin_fold(fold+1)
    tf.random.set_seed(SEED + fold)

    X_tr = X_images_all[tr_idx]
    X_va = X_images_all[va_idx]
    y_tr = y.values[tr_idx]
    y_va = y.values[va_idx]

    cnn_model = build_cnn_model()
    steps = len(X_tr) // cnn_batch_size
    cnn_epochs = 150

    lr_sched = tf.keras.optimizers.schedules.CosineDecay(3e-4, steps * cnn_epochs, alpha=1e-6)
    cnn_model.compile(optimizer=optimizers.Adam(learning_rate=lr_sched),
                      loss=losses.Huber(), metrics=[metrics.MeanAbsoluteError(name='mae')])

    es = callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    cnn_model.fit(augmented_generator(X_tr, y_tr, batch_size=cnn_batch_size),
                  steps_per_epoch=steps, validation_data=(X_va, y_va),
                  epochs=cnn_epochs, callbacks=[es], verbose=0)

    oof_cnn_log[va_idx] = cnn_model.predict(X_va, verbose=0).flatten()
    test_cnn_log += cnn_model.predict(X_images_test, verbose=0).flatten() / N_FOLDS

    fold_mape = mean_absolute_percentage_error(y_raw.values[va_idx], np.expm1(oof_cnn_log[va_idx]))
    progress.end_fold(fold_mape)

oof_cnn_price = np.expm1(oof_cnn_log)
test_cnn_price = np.expm1(test_cnn_log)
cnn_mape = mean_absolute_percentage_error(y_raw, oof_cnn_price)
progress.summary(cnn_mape)

In [ ]:
submission_cnn = pd.DataFrame({
    'House ID': df_test[id_col].astype(int),
    'Price': np.maximum(test_cnn_price, 0)
})
submission_cnn.to_csv('submissions/submission_cnn.csv', index=False)
print('CNN submission opgeslagen: submissions/submission_cnn.csv')
display(submission_cnn.head())

### Hoofdstuk 5: Transfer Learning

We gebruiken EfficientNetB0, voorgetraind op ImageNet (1.2M afbeeldingen, 1000 klassen).
De geleerde visuele representaties (randen, texturen, objecten) zijn direct toepasbaar
op vastgoedafbeeldingen. We bevriezen eerst de backbone en trainen alleen de nieuwe
regressiekop, waarna we de bovenste lagen fine-tunen.

#### 5.1 Strategie
EfficientNetB0 (5.3M params) via Compound Scaling. Fase 1: backbone bevroren, hoge LR. Fase 2: top 30 lagen fine-tunen met lage LR (1e-5) tegen catastrophic forgetting.

In [ ]:
def build_transfer_model(img_size=IMG_SIZE):
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    base_model.trainable = False
    
    inputs = layers.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='linear')(x)
    
    model = models.Model(inputs, outputs, name='efficientnet_transfer')
    return model, base_model

tl_model, tl_base = build_transfer_model()
tl_model.summary(show_trainable=True)

print(f"\nTotaal lagen in backbone: {len(tl_base.layers)}")
print(f"Trainbare parameters (fase 1): {sum(tf.keras.backend.count_params(w) for w in tl_model.trainable_weights):,}")

#### 5.2 K-Fold training (fase 1 + 2 per fold)

In [ ]:
# Transfer Learning K-Fold (fase 1 + 2 per fold)
oof_tl_log = np.zeros(len(df_train))
test_tl_log = np.zeros(len(df_test))

progress = TrainProgress('Transfer Learning', N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(df_train)))):
    progress.begin_fold(fold+1)
    tf.random.set_seed(SEED + fold)

    X_tr = X_images_all[tr_idx]
    X_va = X_images_all[va_idx]
    y_tr = y.values[tr_idx]
    y_va = y.values[va_idx]
    steps = len(X_tr) // 32

    # Model bouwen (vers per fold)
    tl_model, tl_base = build_transfer_model()

    # Fase 1: backbone bevroren
    lr1 = tf.keras.optimizers.schedules.CosineDecay(1e-3, steps * 40, alpha=1e-6)
    tl_model.compile(optimizer=optimizers.Adam(learning_rate=lr1),
                     loss=losses.Huber(), metrics=[metrics.MeanAbsoluteError(name='mae')])
    es1 = callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
    tl_model.fit(augmented_generator(X_tr, y_tr, batch_size=32),
                 steps_per_epoch=steps, validation_data=(X_va, y_va),
                 epochs=40, callbacks=[es1], verbose=0)

    # Fase 2: fine-tuning top 30 lagen
    tl_base.trainable = True
    for layer in tl_base.layers[:max(0, len(tl_base.layers) - 30)]:
        layer.trainable = False
    lr2 = tf.keras.optimizers.schedules.CosineDecay(1e-5, steps * 40, alpha=1e-7)
    tl_model.compile(optimizer=optimizers.Adam(learning_rate=lr2),
                     loss=losses.Huber(), metrics=[metrics.MeanAbsoluteError(name='mae')])
    es2 = callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
    tl_model.fit(augmented_generator(X_tr, y_tr, batch_size=32),
                 steps_per_epoch=steps, validation_data=(X_va, y_va),
                 epochs=40, callbacks=[es2], verbose=0)

    oof_tl_log[va_idx] = tl_model.predict(X_va, verbose=0).flatten()
    test_tl_log += tl_model.predict(X_images_test, verbose=0).flatten() / N_FOLDS

    fold_mape = mean_absolute_percentage_error(y_raw.values[va_idx], np.expm1(oof_tl_log[va_idx]))
    progress.end_fold(fold_mape)

oof_tl_price = np.expm1(oof_tl_log)
test_tl_price = np.expm1(test_tl_log)
tl_mape = mean_absolute_percentage_error(y_raw, oof_tl_price)
progress.summary(tl_mape)

#### 5.3 Resultaten

In [ ]:
print(f"Transfer Learning OOF MAPE: {tl_mape:.4f}")

In [ ]:
submission_tl = pd.DataFrame({
    'House ID': df_test[id_col].astype(int),
    'Price': np.maximum(test_tl_price, 0)
})
submission_tl.to_csv('submissions/submission_transfer_learning.csv', index=False)
print('TL submission opgeslagen: submissions/submission_transfer_learning.csv')
display(submission_tl.head())

### Hoofdstuk 6: Multimodaal Model

Het multimodale model combineert de kracht van twee informatiebronnen:
1. **Tabulaire data** (slaapkamers, oppervlakte, locatie) via een Dense-tak
2. **Afbeeldingen** (visuele kwaliteit, afwerking) via een EfficientNetB0-tak

De features van beide takken worden samengevoegd (**late fusion**) en een
gezamenlijke Dense-kop leert de optimale combinatie voor prijsvoorspelling.

#### 6.1 Architectuur
Image branch (EfficientNetB0) + tabular branch (Dense). Late fusion via Concatenate.
Elk netwerk leert eigen abstracties voordat samenwerking begint.

In [ ]:
def build_multimodal_model(n_tabular_features, img_size=IMG_SIZE):
    img_input = layers.Input(shape=(img_size, img_size, 3), name='image_input')
    
    base_model = EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    
    img_features = base_model(img_input, training=False)
    img_features = layers.GlobalAveragePooling2D()(img_features)
    img_features = layers.Dense(128, activation='relu',
                                kernel_regularizer=tf.keras.regularizers.l2(1e-4))(img_features)
    img_features = layers.BatchNormalization()(img_features)
    img_features = layers.Dropout(0.4)(img_features)
    
    tab_input = layers.Input(shape=(n_tabular_features,), name='tabular_input')
    tab_features = layers.Dense(128, activation='relu',
                                kernel_regularizer=tf.keras.regularizers.l2(1e-4))(tab_input)
    tab_features = layers.BatchNormalization()(tab_features)
    tab_features = layers.Dropout(0.3)(tab_features)
    tab_features = layers.Dense(64, activation='relu',
                                kernel_regularizer=tf.keras.regularizers.l2(1e-4))(tab_features)
    tab_features = layers.BatchNormalization()(tab_features)
    
    merged = layers.Concatenate()([img_features, tab_features])
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(merged)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='linear', name='price_output')(x)
    
    model = models.Model(inputs=[img_input, tab_input], outputs=output, name='multimodal_regressor')
    return model

mm_model = build_multimodal_model(X_train_scaled.shape[1])
mm_model.summary()

#### 6.2 K-Fold training

In [ ]:
def multimodal_generator(X_imgs, X_tabs, y_targets, batch_size=32, datagen=train_datagen):
    n = len(X_imgs)
    while True:
        indices = np.random.permutation(n)
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_idx = indices[start:end]
            batch_imgs = np.array([datagen.random_transform(X_imgs[i]) for i in batch_idx])
            batch_tabs = X_tabs[batch_idx]
            batch_y = y_targets[batch_idx]
            yield {'image_input': batch_imgs, 'tabular_input': batch_tabs}, batch_y

# Multimodaal K-Fold
oof_mm_log = np.zeros(len(df_train))
test_mm_log = np.zeros(len(df_test))
X_tab_all_scaled = scaler.transform(df_train[feature_cols].fillna(median_values))
X_tab_test_scaled = scaler.transform(df_test[feature_cols].fillna(median_values))

progress = TrainProgress('Multimodaal', N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(df_train)))):
    progress.begin_fold(fold+1)
    tf.random.set_seed(SEED + fold)

    X_img_tr = X_images_all[tr_idx]
    X_img_va = X_images_all[va_idx]
    X_tab_tr = X_tab_all_scaled[tr_idx]
    X_tab_va = X_tab_all_scaled[va_idx]
    y_tr = y.values[tr_idx]
    y_va = y.values[va_idx]
    steps = len(X_img_tr) // 32

    mm_model = build_multimodal_model(X_tab_all_scaled.shape[1])
    lr_mm = tf.keras.optimizers.schedules.CosineDecay(3e-4, steps * 100, alpha=1e-6)
    mm_model.compile(optimizer=optimizers.Adam(learning_rate=lr_mm),
                     loss=losses.Huber(), metrics=[metrics.MeanAbsoluteError(name='mae')])

    es = callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    mm_model.fit(
        multimodal_generator(X_img_tr, X_tab_tr, y_tr, batch_size=32),
        steps_per_epoch=steps,
        validation_data=({'image_input': X_img_va, 'tabular_input': X_tab_va}, y_va),
        epochs=100, callbacks=[es], verbose=0)

    oof_mm_log[va_idx] = mm_model.predict(
        {'image_input': X_img_va, 'tabular_input': X_tab_va}, verbose=0).flatten()
    test_mm_log += mm_model.predict(
        {'image_input': X_images_test, 'tabular_input': X_tab_test_scaled}, verbose=0).flatten() / N_FOLDS

    fold_mape = mean_absolute_percentage_error(y_raw.values[va_idx], np.expm1(oof_mm_log[va_idx]))
    progress.end_fold(fold_mape)

oof_mm_price = np.expm1(oof_mm_log)
test_mm_price = np.expm1(test_mm_log)
mm_mape = mean_absolute_percentage_error(y_raw, oof_mm_price)
progress.summary(mm_mape)

In [ ]:
submission_mm = pd.DataFrame({
    'House ID': df_test[id_col].astype(int),
    'Price': np.maximum(test_mm_price, 0)
})
submission_mm.to_csv('submissions/submission_multimodal.csv', index=False)
print('Multimodaal submission opgeslagen: submissions/submission_multimodal.csv')
display(submission_mm.head())

### Ensemble en Finale Submissions

We combineren de voorspellingen van alle modellen in een gewogen ensemble.
De gewichten worden geoptimaliseerd met scipy.optimize om de MAPE te minimaliseren.

In [ ]:
# Ensemble: optimaliseer gewichten op alle 500 OOF voorspellingen
model_names = ['FC', 'CNN', 'TL', 'MM']
oof_preds = [oof_fc_price, oof_cnn_price, oof_tl_price, oof_mm_price]
test_preds = [test_fc_price, test_cnn_price, test_tl_price, test_mm_price]

print("--- OOF Scores per Model ---")
for name, oof in zip(model_names, oof_preds):
    mape = mean_absolute_percentage_error(y_raw, oof)
    print(f"{name:4s}: MAPE = {mape:.4f}")

# Gewichten optimaliseren met scipy
def ens_mape(weights):
    w = np.abs(weights) / np.sum(np.abs(weights))
    blend = sum(wi * p for wi, p in zip(w, oof_preds))
    return mean_absolute_percentage_error(y_raw, blend)

result = minimize(ens_mape, [0.5, 0.05, 0.15, 0.3], method='Nelder-Mead', options={'maxiter': 10000})
opt_w = np.abs(result.x) / np.sum(np.abs(result.x))

print("\n--- Geoptimaliseerde Gewichten ---")
for name, w in zip(model_names, opt_w):
    print(f"{name:4s}: {w:.4f}")

oof_ens = sum(w * p for w, p in zip(opt_w, oof_preds))
ens_mape_val = mean_absolute_percentage_error(y_raw, oof_ens)
print(f"\nEnsemble OOF MAPE: {ens_mape_val:.4f}")

# Finale test ensemble
test_ensemble = sum(w * p for w, p in zip(opt_w, test_preds))
os.makedirs('submissions', exist_ok=True)
submission_ens = pd.DataFrame({
    'House ID': df_test[id_col].astype(int),
    'Price': np.maximum(test_ensemble, 0)
})
submission_ens.to_csv('submissions/submission_ensemble_fc_heavy.csv', index=False)
print(f'\nEnsemble submission opgeslagen: submissions/submission_ensemble_fc_heavy.csv')
display(submission_ens.head())

### Hoofdstuk 7: Conclusie, Synthese en Advies

#### Kritische Reflectie op Experimenten en Bevindingen
We hebben vier verschillende deep learning benaderingen getest: een FC Feed-Forward netwerk op basis van alleen tabeldata, een basaal CNN netwerk met Custom architectuur, Transfer Learning met EfficientNet geselecteerd op ImageNet en ten slotte een fusie hiervan als Multimodaal netwerk. 
Over de hele linie zien we dat de inzet van tabulaire data superieur presteerde en van cruciaal belang was met het voorspellen van de de prijs. Alhoewel de beelden op zichzelf nuttige informatie bevatten om estethische kwaliteit en luxegoederen (die niet de tabel bevonden) konden weergeven, leed het de pure netwerken aan de relatief lage dataset-grootte wat een last was voor het direct from-scratch leren van robuuste weights. Transfer learning overkwam dit heuvel qua low-level gradient optimalisaties enorm (het herkennen van basis in de foto, randen, meubels, patronen), waarna fijntuning via kleine Learning-Rates specifieke details liet inlezen. Echter was de synergie op de Multimodale late-fusion architectuur het absolute hoogtepunt: het gaf de tabel extra vleugels door daadwerkelijk visuele context toe te voegen waar het dense algoritme alleen wellicht de blinde hoek op lag (bijvoorbeeld de staat van de keuken).

#### Aanbevelingen voor Toekomstige Verbeteringen
Voor een toekomstgericht en professioneel doorontwikkeld vervolgmodel geef ik hier nog de volgende aanbevelingen:
1. **Andere Fusie-architecturen**: Hoewel _Late Fusion_ over het algemeen als een robuuste maatstaf fungeert, is het onderzoeken van _Middle/Intermediate fusion_ met eventuele cross-attention blocken uit de transformer theorie absoluut de moeite waard om een beter leermechanisme te ontwikkelen dat dynamisch zwaar weegt op 'welk aspect' tussen de foto/tabel momenteel klopt. 
2. **Afbeeldingen pre-processing / Image cropping**: Aangezien veel huis-afbeeldingen 4 grids aan informatie hadden (4 aparte badkamers, keuken kamers aan de binnenkant) zou een script om bij de data pre-processing een resnet op basis van bounding boxes vier gesplitste losse images te laten trainen kunnen fungeren als aparte bronnen per kamer via Multi-Instance Learning in plaats van als een chaotisch totaalplaatje waar resolutie verloren bij gaat in resize compressie.
3. **Ensemble Technieken**: We zien traditioneel dat gradient boosting (XGBoost/LightGBM) veel beter en meer generaliseert op zuivere tabeldata met dit formaat dan deep learning. Het zou mooi wezen om de feature vectors van de EfficientNet head er simpelweg als extra kolommen en dimensies aan XGBoost of Catboost mee te voeden in samenwerking met de locatiedata, daar verwacht ik een verbluffend lagere validatie fout op.